In [ ]:
from __future__ import absolute_import, division, print_function, unicode_literals

import warnings

warnings.simplefilter("ignore")

# general purpose packages
import pandas as pd
import numpy as np
import os
import json
import time
import re
import csv
import subprocess
import sys

from dotenv import load_dotenv
from pathlib import Path

import scipy.stats as stats
import statsmodels.stats as smstats
from statsmodels.stats.multitest import multipletests

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import sklearn

from multiprocessing import Process, Manager, Pool
import multiprocessing
from functools import partial

from collections import Counter

import seaborn as sns

sns.set()

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

matplotlib.rcParams["backend"] = "Qt5Agg"
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter

from IPython.display import display, Image

from adjustText import adjust_text
import builtins

%matplotlib inline

# for working with sam/bam files
import HTSeq

# for working with fasta files for example
from Bio import SeqIO

# for working with yaml files
import ruamel.yaml

import itertools

In [ ]:
def get_pvalue_star(pval, thr=0.05):
    if thr == 0.05:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.05:
            return "*"
        else:
            return "ns"
    elif thr == 0.1:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.1:
            return "*"
        else:
            return "ns"

In [ ]:
# 1. Load the environment variables
load_dotenv("CRISPR_projects.scicore.env")

# 2. Reconstruct the subdirs dictionary
subdirs = {
    "lab_group_dir": os.getenv("LAB_GROUP_DIR"),
    "main_project_dir": os.getenv("MAIN_PROJECT_DIR"),
    "wf_dir": os.getenv("WF_DIR"),
    "UCSCtracks_dir": os.getenv("UCSC_TRACKS_DIR"),
    "UCSCtracks_trackfiles_dir": os.getenv("UCSC_TRACKFILES_DIR"),
    "UCSCtracks_trackhubs_dir": os.getenv("UCSC_TRACKHUBS_DIR"),
    "human_annotation_dir": os.getenv("HUMAN_ANNOTATION_DIR"),
    "shared_project_dir": os.getenv("SHARED_PROJECT_DIR"),
    "temp_dir": os.getenv("TEMP_DIR"),
    "slurm_dir": os.getenv("SLURM_DIR"),
    "slurm_scripts_dir": os.getenv("SLURM_SCRIPTS_DIR"),
    "figures_dir": os.getenv("FIGURES_DIR"),
    "tables_dir": os.getenv("TABLES_DIR"),
    "fastq_dir": os.getenv("FASTQ_DIR"),
    "metadata_dir": os.getenv("METADATA_DIR"),
    "ext_data_dir": os.getenv("EXT_DATA_DIR"),
    "wf_runs_dir": os.getenv("WF_RUNS_DIR"),
    "HCC_nORFs_dir": os.getenv("HCC_NORFS_DIR"), # specific for HCC translation project, made by Dr. Meric Ataman
}

# 3. Reconstruct the file_paths dictionary
file_paths = {
    "human_genome_file": os.getenv("HUMAN_GENOME_FILE"),
    "human_chrom_sizes_file": os.getenv("HUMAN_CHROM_SIZES_FILE"),
    "human_annotation_file": os.getenv("HUMAN_ANNOTATION_FILE"),
    "human_basic_annotation_file": os.getenv("HUMAN_BASIC_ANNOTATION_FILE"),
    "human_RNAcentral_annotation_file": os.getenv("HUMAN_RNACENTRAL_ANNOTATION_FILE"),
    "human_enriched_annotation_file": os.getenv("HUMAN_ENRICHED_ANNOTATION_FILE"),
    "human_RefSeq_annotation_file": os.getenv("HUMAN_REFSEQ_ANNOTATION_FILE"),
    "human_RefSeq_chrNames": os.getenv("HUMAN_REFSEQ_CHRNAMES"),
    "human_PRErRNA_file": os.getenv("HUMAN_PRERRNA_FILE"),
    "human_polyAsite_atlas": os.getenv("HUMAN_POLYASITE_ATLAS"),
    "human_tandem_PAS": os.getenv("HUMAN_TANDEM_PAS"),
    "lincRNA_annotation_file": os.getenv("LINCRNA_ANNOTATION_FILE"), # specific for HCC translation project, used by Dr. Meric Ataman
    "lincRNA_ORFs": os.getenv("LINCRNA_ORFS"), # specific for HCC translation project, used by Dr. Meric Ataman
    "ORFs_main_annotation_file": os.getenv("ORFS_MAIN_ANNOTATION_FILE"), # specific for HCC translation project, used by Dr. Meric Ataman
}

# 4. Safely create all subdirectories
# Using os.makedirs is highly preferred over os.system('mkdir -p')
# because it avoids opening a subshell and handles permissions gracefully in pure Python.
for path in subdirs.values():
    if path:  # Safety check to ensure the variable was actually found in the .env
        os.makedirs(path, exist_ok=True)

print("Environment loaded and directories verified.")

# Analyzing evolutionary and population genetics features of nORFs vs CDSs related to liver

In [ ]:
# Prepare combined .bed file with all transcriptomic coordinates of nORFs

In [ ]:
transcript_coords_nORF_files = {
    'uORF':{'RiboSeq_evidence':'uorfs_hcc_genome_coords.bed','no_evidence':'unsuccessful_uORF_coordinates.tsv'},
    'dORF':{'RiboSeq_evidence':'dorfs_hcc_genome_coords.bed','no_evidence':'unsuccessful_dORF_coordinates.tsv'},
    'RT':{'RiboSeq_evidence':'rts_hcc_genome_coords.bed','no_evidence':None},
    'lincRNA':{'RiboSeq_evidence':None,'no_evidence':'unsuccessful_lncrna_coordinates.txt'},
}

a = []
for category in transcript_coords_nORF_files.keys():
    for file_type in transcript_coords_nORF_files[category].keys():
        if transcript_coords_nORF_files[category][file_type] is not None:
            file_path = os.path.join(subdirs['HCC_nORFs_dir'],transcript_coords_nORF_files[category][file_type])
            tmp_df = pd.read_csv(file_path, sep='\t', header=None) # expect a three-column tab-delimited file without header
            tmp_df.columns = ['transcript_id','start','end']
            tmp_df['transcript_id'] = tmp_df['transcript_id'].str.replace('_','.') # need to reformat to adhere to namings in .gtf file
            tmp_df['nORF_type'] = category
            tmp_df['subset'] = file_type
            a.append(tmp_df)
nORFs_df = pd.concat(a).reset_index(drop=True) # include so far only nORFs in protein-coding transcripts

# identified lincRNA ORFs were collected separately

lincRNA_ORFs_df = pd.read_csv(file_paths['lincRNA_ORFs'], sep='\t', header=None)
lincRNA_ORFs_df[0] = lincRNA_ORFs_df[0].str.replace('_','.') # need to reformat to adhere to namings in .gtf file

lincRNA_ORFs_df.columns = ['transcript_id','start','end']
lincRNA_ORFs_df['nORF_type'] = 'lincRNA'
lincRNA_ORFs_df['subset'] = 'RiboSeq_evidence'

nORFs_df = pd.concat([nORFs_df,lincRNA_ORFs_df]).reset_index(drop=True) # now we have complete set of nORFs

In [ ]:
nORFs_df['t']=1
nORFs_df.groupby(['nORF_type','subset']).agg({'t':sum}).reset_index().sort_values(['nORF_type','subset'])

In [ ]:
# also add CDSs

CDSs_df = pd.read_csv(os.path.join(subdirs['HCC_nORFs_dir'],'transcript_id_gene_id_CDS.tsv'), sep='\t', header=None)
CDSs_df.columns = ['transcript_id','gene_id','start','end'] # CDS coordinates are: start is the first nt of start codon, and end is (the last nt of stop codon?)
# check that there is just one transcript per gene
CDSs_df['t']=1
gr = CDSs_df.groupby(['gene_id']).agg({'t':sum}).reset_index()
print(f'there are {len(gr.loc[gr['t']>1])} genes with more than one transcript')

In [ ]:
# check that for dORFs, uORFs, and RTs, there are CDSs containing those

CDSs_df['CDS_present'] = 1
tmp_df = pd.merge(nORFs_df,CDSs_df[['transcript_id','CDS_present']],how='left',on='transcript_id')

tmp_df['t'] = 1
tmp_df.loc[tmp_df['CDS_present'].isna()].groupby(['nORF_type','subset']).agg({'t':sum}).reset_index() # this is where respective CDS element is not present

the displayed table only includes lincRNAs, which is expected. It means that for uORFs, dORFs, RTs, we have all respective CDSs

In [ ]:
CDSs_df['custom_transcript_id'] = CDSs_df['transcript_id']+'_CDS_1'+'_'+CDSs_df['start'].astype('str')+'_'+CDSs_df['end'].astype('str')
subset_dict = {'RiboSeq_evidence':1,'no_evidence':0} # for more efficient storage in text files
nORFs_df['custom_transcript_id'] = nORFs_df['transcript_id']+'_'+nORFs_df['nORF_type']+'_'+nORFs_df['subset'].map(subset_dict).astype('str')+'_'+nORFs_df['start'].astype('str')+'_'+nORFs_df['end'].astype('str')

In [ ]:
transcript_coords_df = pd.concat([CDSs_df[['transcript_id','start','end','custom_transcript_id']],
                                 nORFs_df[['transcript_id','start','end','custom_transcript_id']]]).reset_index(drop=True)
len(transcript_coords_df)

In [ ]:
# Define outdir where to write prepared files
outdir = os.path.join(subdirs['tables_dir'],'HCC_ORFs','annotations')
os.makedirs(outdir, exist_ok=True)

transcript_coords_bed_path = os.path.join(outdir,'transcript_coords.12_06_26.bed')
transcript_coords_df.to_csv(transcript_coords_bed_path,sep=str('\t'),header=False,index=None,quoting=csv.QUOTE_NONE)

In [ ]:
# we need to make combined .gtf with both lincRNA and main annotated transcripts from Ensembl

# this is the file that Meric used as a source
gtf = pd.read_csv(file_paths['ORFs_main_annotation_file'],delimiter="\t",index_col=None,header=None,comment='#')
gtf[0] = ('chr'+gtf[0].astype('str')).str.replace('chrMT','chrM') # adhere to GENCODE chromosome names

In [ ]:
lincRNA_gtf = pd.read_csv(file_paths['lincRNA_annotation_file'],delimiter="\t",index_col=None,header=None,comment='#')
lincRNA_gtf = lincRNA_gtf.loc[lincRNA_gtf[0].str.startswith('chr')].reset_index(drop=True) # discard broken lines in this .gtf

In [ ]:
# check that this annotation now has all the transcript ids used in nORFs

lincRNA_gtf['transcript_id'] = lincRNA_gtf[8].str.extract(r'transcript_id "([^"]+)"', expand=False) 

lincRNA_gtf['gtf_present'] = 1
tmp_df = pd.merge(nORFs_df.loc[nORFs_df['nORF_type']=='lincRNA'],
        lincRNA_gtf[['transcript_id','gtf_present']].drop_duplicates(),
         how='left',on = ['transcript_id'])
print(f'{len(tmp_df.loc[tmp_df['gtf_present']!=1])} lincRNA from ORF set dont have respective annotation')

In [ ]:
lincRNA_gtf = lincRNA_gtf.drop(['transcript_id','gtf_present'],axis=1) # not needed further

In [ ]:
gtf_df_compiled = pd.concat([gtf,lincRNA_gtf]).reset_index(drop=True) # appended lincRNA and main annotation

In [ ]:
gtf_df_compiled = gtf_df_compiled.sort_values([0,3]).reset_index(drop=True) # make it directly IGV-compatible

In [ ]:
gtf_df_compiled.to_csv(os.path.join(outdir, 'compiled_genome_annotation.gtf'), sep=str('\t'),header=False,index=None,quoting=csv.QUOTE_NONE)

In [ ]:
nORFs_CDSs_gtf_path = os.path.join(outdir,'nORFs_CDSs.gtf') # this will be output

input_gtf_path = os.path.join(outdir, 'compiled_genome_annotation.gtf')

command = "transcript_to_genomic"+\
" -b "+transcript_coords_bed_path+\
" -g "+input_gtf_path+\
" -o "+nORFs_CDSs_gtf_path+\
" -t $SCRATCH"
print(command)
# On the computational node in bash, activate `zavolab_pyutils` env and then execute the printed command

In [ ]:
print(f'in total we will characterize {len(transcript_coords_df)} ORFs and CDSs')

In [ ]:
# load the prepared .gtf file with proper chromosome names (see above)
from zavolab_pyutils.general import (
    parse_gtf_attributes_into_pd_dataframes,
)

gtf_df, genes_df, exons_df = parse_gtf_attributes_into_pd_dataframes(nORFs_CDSs_gtf_path,
                                                                     gene_type_field = "gene_type",
                                                                     extract_exon_number = True, 
                                                                     extract_gene_name_in_exons = True,
                                                                     verbose=False,
                                                                    )

In [ ]:
# 1. Prepare directories and define tracks
track_dir = subdirs["ext_data_dir"] + "conservation_and_mutations/"
os.makedirs(track_dir, exist_ok=True)

ucsc_base = "http://hgdownload.cse.ucsc.edu/goldenpath/hg38"
broad_base = "https://data.broadinstitute.org/compbio1/PhyloCSFtracks/hg38/latest"
ucsc_gbdb = "http://hgdownload.soe.ucsc.edu/gbdb/hg38"

# Dictionary format: { "TrackName": ("url", "extension") }
tracks_to_download = {
    "PhyloP100way": (f"{ucsc_base}/phyloP100way/hg38.phyloP100way.bw", "bw"),
    "PhastCons100way": (f"{ucsc_base}/phastCons100way/hg38.phastCons100way.bw", "bw"),
    "PhyloP30way": (f"{ucsc_base}/phyloP30way/hg38.phyloP30way.bw", "bw"),
    "PhastCons30way": (f"{ucsc_base}/phastCons30way/hg38.phastCons30way.bw", "bw"),
    
    # Population and Cancer Mutation Tracks (BigBed format)
    "gnomAD_Variants": (f"{ucsc_gbdb}/gnomAD/v3.1.1/genomes.bb", "bb"),
    "TCGA_LIHC_Mutations": (f"{ucsc_gbdb}/gdcCancer/LIHC.bb", "bb"),
    
    # PhyloCSF Power Track
    "PhyloCSF_Power": (f"{broad_base}/PhyloCSFpower.bw", "bw")
}

# Add PhyloCSF frame-specific tracks 
for strand, char in [('plus', '+'), ('minus', '-')]:
    for frame in [1, 2, 3]:
        url_char = "%2B" if char == "+" else "-"
        track_name = f"PhyloCSF{char}{frame}"
        tracks_to_download[track_name] = (f"{broad_base}/PhyloCSF{url_char}{frame}.bw", "bw")

# Generate and print download commands
command = f"mkdir -p {track_dir} && cd {track_dir}\n"
for track_name, (url, ext) in tracks_to_download.items():
    command += f"wget -c -O {track_name}.{ext} {url} \n"

print(command)
# Execute this in bash

In [ ]:
organism = "human"

chrom_sizes = pd.read_csv(file_paths[organism + "_chrom_sizes_file"], sep='\t', header=None, names=['chrom', 'size'])
chrom_sizes_dict = dict(zip(chrom_sizes['chrom'], chrom_sizes['size']))

cds_df = exons_df.copy() # the exons in our case represent coding sequences, CDSs
cds_df['chrom'] = cds_df[0]
cds_df['start_0'] = cds_df[3].astype(int) - 1 # adhere to bed format
cds_df['end_1'] = cds_df[4].astype(int)
cds_df['strand'] = cds_df[6]

# account for strand and genomic frame
def get_phylocsf_track(row):
    chrom = row['chrom']
    if row['strand'] == '+':
        frame = (row['start_0'] % 3) + 1
        return f"PhyloCSF+{frame}"
    elif row['strand'] == '-':
        if chrom in chrom_sizes_dict:
            frame = ((chrom_sizes_dict[chrom] - row['end_1']) % 3) + 1
            return f"PhyloCSF-{frame}"
    return None

cds_df['phylocsf_track'] = cds_df.apply(get_phylocsf_track, axis=1)

bed_jobs_dir = subdirs["temp_dir"] + "conservation_bed_jobs/"
os.makedirs(bed_jobs_dir, exist_ok=True)

# 1. Genome-wide standard tracks
standard_tracks = ["PhyloP100way", 
                   "PhastCons100way", 
                   "PhyloP30way", "PhastCons30way", "PhyloCSF_Power", # comment-out for testing
                   # "gnomAD_Variants", # this is very large
                   "TCGA_LIHC_Mutations", 
                  ]

for track in standard_tracks:
    tmp_df = cds_df[['chrom', 'start_0', 'end_1', 'gene_id']].copy()
    tmp_df['track_name'] = track
    tmp_df.to_csv(f"{bed_jobs_dir}/{track}.bed", sep='\t', header=False, index=False)

# 2. Subset BEDs for frame-specific PhyloCSF scores
tmp_csf = cds_df.dropna(subset=['phylocsf_track']).copy()
for track, group in tmp_csf.groupby('phylocsf_track'):
    group[['chrom', 'start_0', 'end_1', 'gene_id', 'phylocsf_track']].to_csv(
        f"{bed_jobs_dir}/{track}.bed", sep='\t', header=False, index=False
    )
    
print(f"Generated BED files for SLURM jobs in {bed_jobs_dir}")

In [ ]:
score_script = os.path.join(os.getcwd(),"analysis_scripts","compute_track_metrics.py")

sbatch_scripts = []
out_tables_dir = os.path.join(subdirs["tables_dir"],"HCC_ORFs","evol_and_genetics_track_scores")
os.makedirs(out_tables_dir, exist_ok=True)

# add PhyloCSF which depend on frame and strand
all_tracks = standard_tracks + [f"PhyloCSF+{i}" for i in [1,2,3]] + [f"PhyloCSF-{i}" for i in [1,2,3]]

for track in all_tracks:
    sbatch_file_path = subdirs["slurm_scripts_dir"] + f"score_{track}.sbatch"
    
    with open(sbatch_file_path, "w") as f:
        command = f"""#!/bin/bash
#SBATCH --job-name=score_{track}
#SBATCH --time=05:59:00
#SBATCH --qos=6hours
#SBATCH --mem=8G
#SBATCH --cpus-per-task=1
#SBATCH --output={subdirs["slurm_dir"]}/score_{track}.%j.out
#SBATCH --error={subdirs["slurm_dir"]}/score_{track}.%j.err

source activate zavolab_pyutils

python {score_script} \\
    {bed_jobs_dir}/{track}.bed \\
    {track_dir} \\
    {out_tables_dir}/{track}_scores.tsv \\
    {track}
"""
        f.write(command)
    sbatch_scripts.append(sbatch_file_path)

print(";\n".join(["sbatch " + elem for elem in sbatch_scripts]))
# execute these commands from the LOGIN node!

In [ ]:
print(out_tables_dir) # here will be the outputs

In [ ]:
# Aggregate the SLURM output TSVs
all_scores_dfs = []
for track in all_tracks:
    res_path = f"{out_tables_dir}/{track}_scores.tsv"
    if os.path.exists(res_path):
        df = pd.read_csv(res_path, sep='\t')
        all_scores_dfs.append(df)

if all_scores_dfs:
    integration_df = all_scores_dfs[0]
    for df in all_scores_dfs[1:]:
        integration_df = pd.merge(integration_df, df, on="Geneid", how="outer")

    # Combine the strand/frame-specific PhyloCSF outputs into unified columns per gene
    csf_mean_cols = [c for c in integration_df.columns if "PhyloCSF" in c and "Mean" in c]
    csf_med_cols = [c for c in integration_df.columns if "PhyloCSF" in c and "Median" in c]
    
    if csf_mean_cols and csf_med_cols:
        integration_df['PhyloCSF_Mean_Combined'] = integration_df[csf_mean_cols].bfill(axis=1).iloc[:, 0]
        integration_df['PhyloCSF_Median_Combined'] = integration_df[csf_med_cols].bfill(axis=1).iloc[:, 0]
        
        # Drop the frame-specific columns
        integration_df.drop(columns=csf_mean_cols + csf_med_cols, inplace=True)

    # Rename 'Geneid' to match the column name in genes_df from Step 2
    integration_df = integration_df.rename(columns={'Geneid': 'gene_id'})

    # Final Merge into the genes_df parsed from annotation GTF
    final_genes_df = pd.merge(genes_df[[0,3,4,6,'gene_id','gene_name','gene_type']], integration_df, on="gene_id", how="left")
    
    print("Integration of Conservation & Mutation metrics successful!")
    display(final_genes_df.head(3))
else:
    print("No score tables found. Ensure SLURM jobs have completed successfully.")

# also need to calculate the total exonic length for gene id to normalize
exons_df['len'] = exons_df[4]-exons_df[3]
final_genes_df = pd.merge(final_genes_df,
         exons_df.groupby('gene_id').agg({'len':sum}).reset_index(),
         how='left',on='gene_id')

In [ ]:
print(f'we assigned PhyloP for {len(final_genes_df.loc[~final_genes_df['PhyloP100way_Mean'].isna()])}, and PhyloCSF for {len(final_genes_df.loc[~final_genes_df['PhyloCSF_Mean_Combined'].isna()])} elements')
print(f'we have started with {len(transcript_coords_df)} ORFs and CDSs')

In [ ]:
final_genes_df['ORF_type'] = final_genes_df.apply(lambda x:x['gene_id'].split('_')[1],1)
subset_extract_dict = {1:'with_evidence',0:'no_evidence'}
final_genes_df['subset'] = final_genes_df.apply(lambda x:subset_extract_dict[int(x['gene_id'].split('_')[2])],1)
final_genes_df['ORF_category'] = final_genes_df['ORF_type']+';'+final_genes_df['subset']

In [ ]:
from scipy.stats import ks_2samp

hue = 'ORF_category'
# hue_order = ['CDS;with_evidence','uORF;with_evidence','uORF;no_evidence','dORF;with_evidence','dORF;no_evidence','RT;with_evidence','lincRNA;with_evidence','lincRNA;no_evidence']
# palette = ['green','orangered','orange','royalblue','lightblue','darkviolet','deeppink','pink']
hue_order = ['uORF;with_evidence','uORF;no_evidence','dORF;with_evidence','dORF;no_evidence','RT;with_evidence','lincRNA;with_evidence','lincRNA;no_evidence']
palette = ['orangered','orange','royalblue','lightblue','darkviolet','deeppink','pink']

# KS-test target pairs and their reporting color
ks_pairs = [
    ('uORF;with_evidence', 'uORF;no_evidence', 'orangered'),
    ('dORF;with_evidence', 'dORF;no_evidence', 'royalblue'),
    ('lincRNA;with_evidence', 'lincRNA;no_evidence', 'deeppink')
]

# --- Text Layout Configuration per Feature ---
# Adjust x, start_y, alignments, and step direction (+ for upward, - for downward)
text_layout_config = {
    'PhyloP100way_Median':      {'x': 0.05, 'start_y': 0.95, 'ha': 'left',  'va': 'top',    'step': -0.06}, # Top-Left
    'PhastCons100way_Median':   {'x': 0.95, 'start_y': 0.05, 'ha': 'right',  'va': 'bottom', 'step': 0.06},  # Bottom-right
    'PhyloP30way_Median':       {'x': 0.95, 'start_y': 0.05, 'ha': 'right',  'va': 'bottom', 'step': 0.06}, # Bottom-right
    'PhastCons30way_Median':    {'x': 0.95, 'start_y': 0.05, 'ha': 'right',  'va': 'bottom', 'step': 0.06},  # Bottom-right
    'PhyloCSF_Median_Combined': {'x': 0.05, 'start_y': 0.95, 'ha': 'left', 'va': 'top', 'step': -0.06}   # Top-Left
}

features = ['PhyloP100way_Median','PhastCons100way_Median','PhyloP30way_Median','PhastCons30way_Median','PhyloCSF_Median_Combined']
feature_labels = ['PhyloP Median\n100 vertebrates','PhastCons Median\n100 vertebrates','PhyloP Median\n30 primates','PhastCons Median\n30 primates','PhyloCSF Median\n58-mammals']
xlims = [(-1,1),None,(-0.5,1.5),None,None]

sns.set(font_scale=1, style="white")
fig, axes = plt.subplots(1, len(features), sharey=True, sharex=False, figsize=(2.8*len(features),3.2))

for k, feature in enumerate(features):
    data = final_genes_df.loc[~final_genes_df[feature].isna()].copy()
    print(f'{len(data)} ORFs for '+feature)
    
    ax = sns.ecdfplot(ax = axes[k],
                      data = data, x = feature, 
                      hue = hue, hue_order = hue_order, palette = palette,
                      alpha=1,
                     )
    
    ax.set(title = '', xlabel = feature_labels[k], ylabel = ('eCDF' if k==0 else ''))
    if xlims[k] is not None:
        ax.set(xlim = xlims[k])
    ax.tick_params(bottom=True, left=(True if k==0 else False))
    
    # --- Dynamic KS-test Annotation ---
    cfg = text_layout_config.get(feature, {'x': 0.95, 'start_y': 0.05, 'ha': 'right', 'va': 'bottom', 'step': 0.06})
    y_pos = cfg['start_y']
    
    for cat_with, cat_no, color in ks_pairs:
        dist_with = data.loc[data[hue] == cat_with, feature]
        dist_no = data.loc[data[hue] == cat_no, feature]
        
        if len(dist_with) > 0 and len(dist_no) > 0:
            stat, p_val = ks_2samp(dist_with, dist_no)
            
            if p_val < 0.001:
                p_text = f"p = {p_val:.2e}"
            else:
                p_text = f"p = {p_val:.3f}"
                
            label = cat_with.split(';')[0]
            text_str = f"{label}: {p_text}"
            
            ax.text(
                cfg['x'], y_pos, text_str, 
                color=color, 
                transform=ax.transAxes, 
                ha=cfg['ha'], 
                va=cfg['va'], 
                fontsize=8,
                weight='semibold'
            )
            # Adjust the vertical position for the next category line
            y_pos += cfg['step']
    # ----------------------------------

    if k < len(features)-1:
        ax.legend_.remove()
    else:
        sns.move_legend(
                    ax, 
                    loc="upper left", 
                    bbox_to_anchor=(1.05, 1), 
                    borderaxespad=0.0, 
                    title='ORF category'
                )

savefig_path = os.path.join(subdirs['figures_dir'], 'HCC_ORFs', 'evol_scores.nORFs_only.pdf')
dir_path = Path(savefig_path).parent
dir_path.mkdir(parents=True, exist_ok=True)

fig.tight_layout(pad=0.5)
fig.savefig(savefig_path, bbox_inches="tight", dpi=600)

now bin all ORFs into groups by their exonic lengths

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

# --- Stratifying Length Confounder (Value-based Binning) ---
bin_edges = [0, 50, 100, 300, np.inf]
final_genes_df['len_bin'] = pd.cut(final_genes_df['len'], bins=bin_edges)

unique_bins = sorted(final_genes_df['len_bin'].dropna().unique())
bin_labels = {
    b: f"{max(0, int(b.left))}-{int(b.right) if b.right != np.inf else 'max'}" 
    for b in unique_bins
}

# --- Core Setup Configuration ---
hue = 'ORF_category'
hue_order = ['uORF;with_evidence','uORF;no_evidence','dORF;with_evidence','dORF;no_evidence','RT;with_evidence','lincRNA;with_evidence','lincRNA;no_evidence']
palette = ['orangered','orange','royalblue','lightblue','darkviolet','deeppink','pink']

ks_pairs = [
    ('uORF;with_evidence', 'uORF;no_evidence', 'orangered'),
    ('dORF;with_evidence', 'dORF;no_evidence', 'royalblue'),
    ('lincRNA;with_evidence', 'lincRNA;no_evidence', 'deeppink')
]

text_layout_config = {
    'PhyloP100way_Median':      {'x': 0.95, 'start_y': 0.05, 'ha': 'right', 'va': 'bottom', 'step': 0.06}, 
    'PhastCons100way_Median':   {'x': 0.95, 'start_y': 0.05, 'ha': 'right', 'va': 'bottom', 'step': 0.06},  
    'PhyloP30way_Median':       {'x': 0.05, 'start_y': 0.95, 'ha': 'left',  'va': 'top',    'step': -0.06}, 
    'PhastCons30way_Median':    {'x': 0.95, 'start_y': 0.05, 'ha': 'right', 'va': 'bottom', 'step': 0.06},  
    'PhyloCSF_Median_Combined': {'x': 0.05, 'start_y': 0.95, 'ha': 'left',  'va': 'top',    'step': -0.06}   
}

features = ['PhastCons100way_Median','PhastCons30way_Median','PhyloCSF_Median_Combined']
feature_labels = ['PhastCons Median\n100 vertebrates','PhastCons Median\n30 primates','PhyloCSF Median\n58-mammals']
xlims = [None, None, None]

# =========================================================================
# NEW: Global Proportions Normalization (Across Bins per Category)
# =========================================================================
# 1. Get raw counts grid (Rows: Category, Columns: Length Bins)
counts_matrix = final_genes_df.groupby([hue, 'len_bin'], observed=False).size().unstack(fill_value=0)

# 2. Divide each row by its total sum (Normalizes across length bins for every category)
normalized_matrix = counts_matrix.div(counts_matrix.sum(axis=1), axis=0)

# 3. Flatten back into a long-form DataFrame for Seaborn mapping
global_prop_df = normalized_matrix.stack().reset_index(name='proportion')


# --- Initialize 2D Subplot Grid ---
num_rows = len(unique_bins)
num_cols = len(features) + 1  
sns.set(font_scale=1, style="white")

fig, axes = plt.subplots(
    nrows=num_rows, 
    ncols=num_cols, 
    sharey=False,       
    sharex=False,       
    figsize=(3.2 * num_cols, 2.2 * num_rows)
)

# --- Grid Traversal Loop ---
for r, bin_val in enumerate(unique_bins):
    row_data = final_genes_df.loc[final_genes_df['len_bin'] == bin_val]
    row_range_str = bin_labels[bin_val]
    
    # =========================================================================
    # COLUMN 0: Relative Proportion Barplots (Corrected Normalization)
    # =========================================================================
    ax_count = axes[r, 0]
    
    # Filter precalculated global proportions for just this length tier row
    prop_df = global_prop_df[global_prop_df['len_bin'] == bin_val]
    
    sns.barplot(
        ax=ax_count, data=prop_df, x=hue, y='proportion', order=hue_order, palette=palette
    )
    
    ax_count.set_ylim(0, 1.0)
    ax_count.set_ylabel(f"Proportion\n(Len: {row_range_str} bp)", fontsize=9)
    ax_count.set_xlabel("" if r == num_rows - 1 else "")
    ax_count.set_xticklabels([]) 
    ax_count.tick_params(bottom=False, left=True)
    
    # =========================================================================
    # COLUMNS 1 to N: Evolutionary Metric eCDF Subplots
    # =========================================================================
    for c_feat, feature in enumerate(features):
        c = c_feat + 1  
        ax = axes[r, c]
        data = row_data.loc[~row_data[feature].isna()].copy()
        
        sns.ecdfplot(
            ax=ax, data=data, x=feature, 
            hue=hue, hue_order=hue_order, palette=palette, alpha=1
        )
        
        ax.set_ylim(-0.02, 1.02)
        ax.set_ylabel("eCDF" if c == 1 else "")
        ax.set_xlabel(feature_labels[c_feat] if r == num_rows - 1 else "")
        
        if xlims[c_feat] is not None:
            ax.set(xlim=xlims[c_feat])
            
        ax.tick_params(bottom=True, left=True)
        
        # --- Context-Specific KS-test Calculations ---
        cfg = text_layout_config.get(feature, {'x': 0.95, 'start_y': 0.05, 'ha': 'right', 'va': 'bottom', 'step': 0.06})
        y_pos = cfg['start_y']
        
        for cat_with, cat_no, color in ks_pairs:
            dist_with = data.loc[data[hue] == cat_with, feature]
            dist_no = data.loc[data[hue] == cat_no, feature]
            
            if len(dist_with) > 5 and len(dist_no) > 5:
                stat, p_val = ks_2samp(dist_with, dist_no)
                p_text = f"p = {p_val:.2e}" if p_val < 0.001 else f"p = {p_val:.3f}"
                label = cat_with.split(';')[0]
                
                ax.text(
                    cfg['x'], y_pos, f"{label}: {p_text}", 
                    color=color, transform=ax.transAxes, 
                    ha=cfg['ha'], va=cfg['va'], 
                    fontsize=7.5, weight='semibold'
                )
                y_pos += cfg['step']
                
        # --- Legend Sanitization ---
        if r == 0 and c == num_cols - 1:
            sns.move_legend(
                ax, loc="upper left", bbox_to_anchor=(1.05, 1), 
                borderaxespad=0.0, title='ORF category'
            )
        else:
            if ax.legend_ is not None:
                ax.legend_.remove()

# --- File Serialization ---
savefig_path = os.path.join(subdirs['figures_dir'], 'HCC_ORFs', 'evol_scores.nORFs_only.stratified_by_ORF_length.pdf')
Path(savefig_path).parent.mkdir(parents=True, exist_ok=True)

fig.tight_layout(pad=0.4)
fig.savefig(savefig_path, bbox_inches="tight", dpi=600)

In [ ]:
# check dependency on len 

In [ ]:
final_genes_df.head()